# 06_burst — Langkah 17 (pengganti CiteSpace)
Burst detection Kleinberg (2003), versi batched dua state, seperti yang dipakai CiteSpace. Jalankan Sel 1, 2, lalu 3.

## Sel 1 — SEL PEMBUKA (Python)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%load_ext rpy2.ipython
!apt-get -qq update > /dev/null
!apt-get -qq install -y libpoppler-cpp-dev libglpk-dev libxml2-dev libcurl4-openssl-dev libssl-dev libfontconfig1-dev libharfbuzz-dev libfribidi-dev libfreetype6-dev libpng-dev libtiff5-dev libjpeg-dev > /dev/null
print('Sel 1 selesai')

## Sel 2 — SEL PEMBUKA (R)

In [ ]:
%%R
proj <- "/content/drive/MyDrive/bibliometrik-cxr"
lib  <- file.path(proj, "Rlib")
dir.create(lib, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(lib, .libPaths()))
codename <- system("lsb_release -cs", intern = TRUE)
options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", codename)))
options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
  paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
pkgs <- c("bibliometrix", "UpSetR", "readxl", "writexl", "ggplot2", "htmlwidgets")
baru <- pkgs[!pkgs %in% rownames(installed.packages())]
if (length(baru)) install.packages(baru, lib = lib)
setwd(proj)
suppressPackageStartupMessages(library(bibliometrix))
cat("Folder kerja:", getwd(), "\nbibliometrix", as.character(packageVersion("bibliometrix")), "siap\n")

## Sel 3 — Burst detection author keywords

In [ ]:
%%R
library(writexl); library(ggplot2)
M <- readRDS("03_clean/M.rds")

sin <- c(
  "PSEUDO-LABELING;PSEUDO LABELING;PSEUDO-LABELLING;PSEUDO LABEL;PSEUDO-LABEL;PSEUDO LABELS;PSEUDO-LABELS",
  "CHEST X-RAY;CHEST X-RAYS;CXR;CHEST X-RAY (CXR);CHEST RADIOGRAPH;CHEST RADIOGRAPHY;CHEST RADIOGRAPHS;CHEST X-RAY IMAGES;CHEST X-RAY IMAGE",
  "SEMI-SUPERVISED LEARNING;SEMI SUPERVISED LEARNING;SEMISUPERVISED LEARNING;SSL;SEMI-SUPERVISED;SEMI-SUPERVISED DEEP LEARNING;SEMI-SUPERVISED CLASSIFICATION",
  "CLASS IMBALANCE;IMBALANCED DATA;DATA IMBALANCE;IMBALANCED LEARNING;IMBALANCED DATASET;CLASS IMBALANCE CLASSIFICATION",
  "MULTI-LABEL CLASSIFICATION;MULTILABEL CLASSIFICATION;MULTI LABEL CLASSIFICATION;MULTI-LABEL",
  "COVID-19;COVID 19;COVID;SARS-COV-2;CORONAVIRUS",
  "MEDICAL IMAGE SEGMENTATION;SEGMENTATION;IMAGE SEGMENTATION",
  "MEDICAL IMAGE CLASSIFICATION;IMAGE CLASSIFICATION;CLASSIFICATION;CHEST X-RAY CLASSIFICATION",
  "COMPUTER AIDED DIAGNOSIS;COMPUTER-AIDED DIAGNOSIS")
hapus <- c("DEEP LEARNING", "MACHINE LEARNING", "ARTIFICIAL INTELLIGENCE",
           "SEMI-SUPERVISED LEARNING", "CHEST X-RAY", "MEDICAL IMAGING", "MEDICAL IMAGE")

norm_kw <- function(x) {
  if (is.na(x)) return(character(0))
  k <- toupper(trimws(unlist(strsplit(x, ";")))); k <- k[k != ""]
  for (s in sin) { v <- strsplit(s, ";")[[1]]; k[k %in% v] <- v[1] }
  setdiff(unique(k), hapus)
}
kw <- lapply(M$DE, norm_kw)
years <- 2018:2026
d <- as.numeric(table(factor(M$PY, levels = years)))
freq <- table(unlist(kw))
terms <- names(freq[freq >= 3])
cat("Istilah yang diuji (frekuensi >= 3):", length(terms), "\n")

# Kleinberg (2003) batched burst detection, dua state (sama dengan CiteSpace)
burst_states <- function(r, d, s = 2, gamma = 1) {
  n <- length(r); p0 <- sum(r) / sum(d); p1 <- min(s * p0, 0.9999)
  cost <- function(p) -(r * log(p) + (d - r) * log(1 - p))
  c0 <- cost(p0); c1 <- cost(p1); tau <- gamma * log(n)
  C <- matrix(Inf, 2, n); B <- matrix(0L, 2, n)
  C[1, 1] <- c0[1]; C[2, 1] <- c1[1] + tau
  for (t in 2:n) {
    a <- C[1, t - 1]; b <- C[2, t - 1]
    if (a <= b) { C[1, t] <- a + c0[t]; B[1, t] <- 1L } else { C[1, t] <- b + c0[t]; B[1, t] <- 2L }
    if (a + tau <= b) { C[2, t] <- a + tau + c1[t]; B[2, t] <- 1L } else { C[2, t] <- b + c1[t]; B[2, t] <- 2L }
  }
  st <- integer(n); st[n] <- which.min(C[, n])
  for (t in n:2) st[t - 1] <- B[st[t], t]
  list(st = st, gain = c0 - c1)
}

hasil <- list()
for (tm in terms) {
  r <- sapply(years, function(y) sum(M$PY == y & sapply(kw, function(k) tm %in% k)))
  b <- burst_states(r, d)
  runs <- rle(b$st == 2); ends <- cumsum(runs$lengths); starts <- ends - runs$lengths + 1
  for (i in which(runs$values)) {
    hasil[[length(hasil) + 1]] <- data.frame(istilah = tolower(tm), frekuensi = sum(r),
      mulai = years[starts[i]], selesai = years[ends[i]],
      kekuatan = round(sum(b$gain[starts[i]:ends[i]]), 2))
  }
}
bt <- do.call(rbind, hasil)
bt <- bt[order(bt$mulai, -bt$kekuatan), ]
print(bt, row.names = FALSE)
write_xlsx(bt, "05_figures/burst_keywords.xlsx")

if (!is.null(bt) && nrow(bt) > 0) {
  bt$istilah <- factor(bt$istilah, levels = rev(unique(bt$istilah)))
  g <- ggplot(bt) +
    geom_segment(aes(x = 2017.5, xend = 2026.5, y = istilah, yend = istilah), colour = "grey80", linewidth = 3) +
    geom_segment(aes(x = mulai - 0.5, xend = selesai + 0.5, y = istilah, yend = istilah), colour = "#c0392b", linewidth = 3) +
    geom_text(aes(x = 2026.9, y = istilah, label = kekuatan), hjust = 0, size = 3) +
    scale_x_continuous(breaks = years, limits = c(2017.5, 2027.5)) +
    labs(x = NULL, y = NULL, title = "Keyword bursts (Kleinberg, 2003)") +
    theme_minimal(base_size = 11) + theme(panel.grid = element_blank())
  ggsave("05_figures/burst_keywords.png", g, width = 9, height = 0.35 * nrow(bt) + 1.5, dpi = 300)
}
cat("Selesai\n")

In [ ]:
%%R
# Sel 4 — Uji sensitivitas burst: frekuensi >= 2, gamma = 0.5
library(writexl); library(ggplot2)
M <- readRDS("03_clean/M.rds")

sin <- c(
  "PSEUDO-LABELING;PSEUDO LABELING;PSEUDO-LABELLING;PSEUDO LABEL;PSEUDO-LABEL;PSEUDO LABELS;PSEUDO-LABELS",
  "CHEST X-RAY;CHEST X-RAYS;CXR;CHEST X-RAY (CXR);CHEST RADIOGRAPH;CHEST RADIOGRAPHY;CHEST RADIOGRAPHS;CHEST X-RAY IMAGES;CHEST X-RAY IMAGE",
  "SEMI-SUPERVISED LEARNING;SEMI SUPERVISED LEARNING;SEMISUPERVISED LEARNING;SSL;SEMI-SUPERVISED;SEMI-SUPERVISED DEEP LEARNING;SEMI-SUPERVISED CLASSIFICATION",
  "CLASS IMBALANCE;IMBALANCED DATA;DATA IMBALANCE;IMBALANCED LEARNING;IMBALANCED DATASET;CLASS IMBALANCE CLASSIFICATION",
  "MULTI-LABEL CLASSIFICATION;MULTILABEL CLASSIFICATION;MULTI LABEL CLASSIFICATION;MULTI-LABEL",
  "COVID-19;COVID 19;COVID;SARS-COV-2;CORONAVIRUS",
  "MEDICAL IMAGE SEGMENTATION;SEGMENTATION;IMAGE SEGMENTATION",
  "MEDICAL IMAGE CLASSIFICATION;IMAGE CLASSIFICATION;CLASSIFICATION;CHEST X-RAY CLASSIFICATION",
  "COMPUTER AIDED DIAGNOSIS;COMPUTER-AIDED DIAGNOSIS")
hapus <- c("DEEP LEARNING", "MACHINE LEARNING", "ARTIFICIAL INTELLIGENCE",
           "SEMI-SUPERVISED LEARNING", "CHEST X-RAY", "MEDICAL IMAGING", "MEDICAL IMAGE")

norm_kw <- function(x) {
  if (is.na(x)) return(character(0))
  k <- toupper(trimws(unlist(strsplit(x, ";")))); k <- k[k != ""]
  for (s in sin) { v <- strsplit(s, ";")[[1]]; k[k %in% v] <- v[1] }
  setdiff(unique(k), hapus)
}
kw <- lapply(M$DE, norm_kw)
years <- 2018:2026
d <- as.numeric(table(factor(M$PY, levels = years)))
freq <- table(unlist(kw))

MIN_FREQ <- 2      # parameter sensitivitas (utama: 3)
GAMMA    <- 0.5    # parameter sensitivitas (utama: 1)

terms <- names(freq[freq >= MIN_FREQ])
cat("Istilah yang diuji (frekuensi >=", MIN_FREQ, "):", length(terms), "\n")

burst_states <- function(r, d, s = 2, gamma = 1) {
  n <- length(r); p0 <- sum(r) / sum(d); p1 <- min(s * p0, 0.9999)
  cost <- function(p) -(r * log(p) + (d - r) * log(1 - p))
  c0 <- cost(p0); c1 <- cost(p1); tau <- gamma * log(n)
  C <- matrix(Inf, 2, n); B <- matrix(0L, 2, n)
  C[1, 1] <- c0[1]; C[2, 1] <- c1[1] + tau
  for (t in 2:n) {
    a <- C[1, t - 1]; b <- C[2, t - 1]
    if (a <= b) { C[1, t] <- a + c0[t]; B[1, t] <- 1L } else { C[1, t] <- b + c0[t]; B[1, t] <- 2L }
    if (a + tau <= b) { C[2, t] <- a + tau + c1[t]; B[2, t] <- 1L } else { C[2, t] <- b + c1[t]; B[2, t] <- 2L }
  }
  st <- integer(n); st[n] <- which.min(C[, n])
  for (t in n:2) st[t - 1] <- B[st[t], t]
  list(st = st, gain = c0 - c1)
}

hasil <- list()
for (tm in terms) {
  r <- sapply(years, function(y) sum(M$PY == y & sapply(kw, function(k) tm %in% k)))
  b <- burst_states(r, d, gamma = GAMMA)
  runs <- rle(b$st == 2); ends <- cumsum(runs$lengths); starts <- ends - runs$lengths + 1
  for (i in which(runs$values)) {
    hasil[[length(hasil) + 1]] <- data.frame(istilah = tolower(tm), frekuensi = sum(r),
      mulai = years[starts[i]], selesai = years[ends[i]],
      kekuatan = round(sum(b$gain[starts[i]:ends[i]]), 2))
  }
}
bt <- do.call(rbind, hasil)
bt <- bt[order(bt$mulai, -bt$kekuatan), ]
print(bt, row.names = FALSE)
write_xlsx(bt, "05_figures/burst_keywords_sensitivitas.xlsx")

if (!is.null(bt) && nrow(bt) > 0) {
  bt$istilah <- factor(bt$istilah, levels = rev(unique(bt$istilah)))
  g <- ggplot(bt) +
    geom_segment(aes(x = 2017.5, xend = 2026.5, y = istilah, yend = istilah), colour = "grey80", linewidth = 3) +
    geom_segment(aes(x = mulai - 0.5, xend = selesai + 0.5, y = istilah, yend = istilah), colour = "#c0392b", linewidth = 3) +
    geom_text(aes(x = 2026.9, y = istilah, label = kekuatan), hjust = 0, size = 3) +
    scale_x_continuous(breaks = years, limits = c(2017.5, 2027.5)) +
    labs(x = NULL, y = NULL, title = "Keyword bursts, sensitivity (min freq 2, gamma 0.5)") +
    theme_minimal(base_size = 11) + theme(panel.grid = element_blank())
  ggsave("05_figures/burst_keywords_sensitivitas.png", g, width = 9, height = 0.35 * nrow(bt) + 1.5, dpi = 300)
}
cat("Selesai\n")